In [1]:
import sys
import os

sys.path.append(
    os.path.abspath(".")
)

In [2]:
sys.path.append(
    os.path.abspath("..")
)

In [3]:
from src.datasets.brats_dataset import create_file_list
from src.config import DATA_DIR
from src.transforms.preprocessing import (
    train_transform,
    val_transform
)

In [4]:
files = create_file_list(DATA_DIR)

print(len(files))

369


In [5]:
from sklearn.model_selection import train_test_split

files = create_file_list(DATA_DIR)

# 70% train
train_files, temp_files = train_test_split(
    files,
    test_size=0.30,
    random_state=42,
    shuffle=True
)

# 15% val, 15% test
val_files, test_files = train_test_split(
    temp_files,
    test_size=0.50,
    random_state=42,
    shuffle=True
)

print("Train:", len(train_files))
print("Val  :", len(val_files))
print("Test :", len(test_files))

Train: 258
Val  : 55
Test : 56


In [6]:
from monai.data import Dataset

train_ds = Dataset(
    data=train_files,
    transform=train_transform
)

val_ds = Dataset(
    data=val_files,
    transform=val_transform
)

test_ds = Dataset(
    data=test_files,
    transform=val_transform
)

In [7]:
from src.graphs.create_graph_dataset import create_graph_dataset

In [8]:
train_graphs = create_graph_dataset(train_ds)

val_graphs = create_graph_dataset(val_ds)

test_graphs = create_graph_dataset(test_ds)

  0%|          | 0/258 [00:00<?, ?it/s]d:\Internship\brats2020\src\graphs\graph_dataset.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  edge_index = torch.tensor(
  0%|          | 1/258 [00:07<31:38,  7.39s/it]d:\Internship\brats2020\src\graphs\graph_dataset.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  edge_index = torch.tensor(
  1%|          | 2/258 [00:11<23:01,  5.40s/it]d:\Internship\brats2020\src\graphs\graph_dataset.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  edge_index = torch.tensor(
  1%|          | 3/258 [

In [9]:
from torch_geometric.loader import DataLoader

train_loader = DataLoader(
    train_graphs,
    batch_size=1,
    shuffle=True
)

val_loader = DataLoader(
    val_graphs,
    batch_size=1,
    shuffle=False
)

test_loader = DataLoader(
    test_graphs,
    batch_size=1,
    shuffle=False
)

In [10]:
from train.train_gat import train_gat

model = train_gat(
    train_loader,
    val_loader,
    epochs=300,
    lr=5e-4,
    hidden=64
)


Training Class Distribution:
Counter({np.int64(0): 813671, np.int64(2): 40256, np.int64(3): 8720, np.int64(1): 8103})

Class Weights:
tensor([ 0.2675, 26.8650,  5.4076, 24.9642])
Epoch 000 | Train Loss: 0.8749 | Train Acc: 0.7732 | Val Loss: 0.5692 | Val Acc: 0.8724
Epoch 010 | Train Loss: 0.6098 | Train Acc: 0.8703 | Val Loss: 0.4327 | Val Acc: 0.9481
Epoch 020 | Train Loss: 0.5802 | Train Acc: 0.8809 | Val Loss: 0.4149 | Val Acc: 0.9523
Epoch 030 | Train Loss: 0.5676 | Train Acc: 0.8879 | Val Loss: 0.4086 | Val Acc: 0.9599
Epoch 040 | Train Loss: 0.5594 | Train Acc: 0.8907 | Val Loss: 0.4123 | Val Acc: 0.9562
Epoch 050 | Train Loss: 0.5540 | Train Acc: 0.8932 | Val Loss: 0.4402 | Val Acc: 0.9712

Early stopping at epoch 51

Training Complete
Best Validation Accuracy: 0.9740


In [11]:
from evaluate_gat import evaluate_gat

evaluate_gat(
    model,
    test_loader
)


===== Evaluation Results =====
Accuracy  : 0.9732282549417539
Precision : 0.6314209858658411
Recall    : 0.7001079678642448
F1 Score  : 0.6071018112174168

Confusion Matrix:

[[856544     64  13117   3234]
 [   387   1147   1781    406]
 [  3157    208  10827   1010]
 [   284     56    252   2350]]

Classification Report:

              precision    recall  f1-score   support

           0       1.00      0.98      0.99    872959
           1       0.78      0.31      0.44      3721
           2       0.42      0.71      0.53     15202
           3       0.34      0.80      0.47      2942

    accuracy                           0.97    894824
   macro avg       0.63      0.70      0.61    894824
weighted avg       0.98      0.97      0.98    894824


===== Dice Scores =====
Class 0: 0.9883
Class 1: 0.4415
Class 2: 0.5259
Class 3: 0.4727

===== BraTS Region Dice Scores =====
WT (Whole Tumor)      : 0.6406
TC (Tumor Core)       : 0.5231
ET (Enhancing Tumor)  : 0.4727
